# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one content item/page at the current snapshot. The dataset contains 30,000 content items and provides search and engagement measurements over rolling windows, including 90-day totals and separate last-30-day and previous-30-day measures. There is no explicit calendar date column in this CSV, so I will describe the data using these available measurement windows rather than claiming exact observation dates.

In [3]:
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nWindow-related fields:")
print([
    col for col in df.columns
    if "90d" in col or "30d" in col or "days_" in col
])

Rows: 30000
Columns: 44

Window-related fields:
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'days_since_last_update']


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

Shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Data types:
content_id                 object
client_id                  object
search_volume             float64
competition               float64
competition_level          object
cpc              

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [2]:
print("\nPossible date/time columns:")
for col in df.columns:
    if "date" in col.lower() or "time" in col.lower():
        print(col, "->", df[col].dtype)


Possible date/time columns:
days_since_last_update -> int64


## 2. Fields: feature / label / context / excluded

Features: search_volume, competition, cpc, word_count, char_count, impressions_90d, clicks_90d, pageviews_90d, sessions_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d, days_with_impressions, days_with_sessions, impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, sessions_prev_30d, content_age_days, days_since_last_update, ctr, avg_position, engagement_rate, scroll_rate, ai_traffic_pct.

Label/proxy: A review-priority proxy will be defined later using measurable content-performance signals. I will not treat content_id or client_id as a predictive target.

Context: content_type, main_intent, competition_level, provider_used, model_used, age_tier, age_tier_order, freshness_tier, word_count_tier, char_count_tier, impression_tier, position_tier, trend_direction, and trend_pct provide context about the content and its observed performance.

Excluded: content_id and client_id will be excluded from modeling because they identify records or clients rather than representing generalizable content-performance signals. I will also avoid using fields that would create target leakage once the final review-priority proxy is defined.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d",
    "sessions_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct"
]

context = [
    "content_type", "main_intent", "competition_level",
    "provider_used", "model_used", "age_tier", "age_tier_order",
    "freshness_tier", "word_count_tier", "char_count_tier",
    "impression_tier", "position_tier", "trend_direction", "trend_pct"
]

excluded = ["content_id", "client_id"]

print("Feature fields:", len(features))
print("Context fields:", len(context))
print("Excluded fields:", len(excluded))
print("Total classified:", len(features) + len(context) + len(excluded))

Feature fields: 27
Context fields: 14
Excluded fields: 2
Total classified: 43


## 3. Verify it with queries (grain, counts, missing values, windows)

The dataset contains 30,000 rows and 44 columns. I will verify the row count, uniqueness of content_id, missing values, and the availability of the 90-day and 30-day measurement fields. These checks help confirm the expected grain and whether the fields required for the lane are usable.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("=== Grain check ===")
print("Rows:", len(df))
print("Unique content_id:", df["content_id"].nunique())

print("\n=== Missing values ===")
missing = df[features + context + excluded].isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\n=== Required time-window fields ===")
window_fields = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print(df[window_fields].describe().T[["count", "min", "max"]])

=== Grain check ===
Rows: 30000
Unique content_id: 30000

=== Missing values ===
provider_used        21438
word_count            7699
char_count_tier       7699
word_count_tier       7699
char_count            7699
model_used            5733
trend_pct             3388
competition_level     2610
search_volume         2468
competition           2468
cpc                   2468
main_intent           2374
scroll_rate            125
dtype: int64

=== Required time-window fields ===
                        count  min       max
impressions_90d       30000.0  1.0  517715.0
clicks_90d            30000.0  0.0    4178.0
sessions_90d          30000.0  1.0    4345.0
impressions_last_30d  30000.0  0.0  238796.0
clicks_last_30d       30000.0  0.0    1176.0
sessions_last_30d     30000.0  0.0    1081.0
impressions_prev_30d  30000.0  0.0  218786.0
clicks_prev_30d       30000.0  0.0    1627.0
sessions_prev_30d     30000.0  0.0    4247.0


## 4. Data limits

This dataset provides a current content-level snapshot with aggregated 90-day and 30-day windows, so it cannot establish exact day-by-day historical behavior or causal effects. The last-30-day and previous-30-day windows may overlap with the broader 90-day history. The dataset also does not contain enough information to explain every reason for a performance change, and it should not be used to claim that a particular content change caused an outcome. Any model result will remain directional decision-support based on the available measurements.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Data limits:")
print("- No explicit calendar date column is available.")
print("- Performance is provided through aggregated 90-day and 30-day windows.")
print("- The dataset cannot establish causal effects.")
print("- Results should be treated as directional decision-support.")

Data limits:
- No explicit calendar date column is available.
- Performance is provided through aggregated 90-day and 30-day windows.
- The dataset cannot establish causal effects.
- Results should be treated as directional decision-support.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.